Lendo do bucket e analisando o conteúdo

In [45]:
import boto3
import pandas as pd

bucket = "desafio-sprint4-vinicius"
arquivo = "canais-de-programacao-de-programadoras-ativos-credenciados.csv"
s3 = boto3.client(
    "s3",
    aws_access_key_id="xxxxxxxx",
    aws_secret_access_key="xxxxxxxx",
    aws_session_token="xxxxxxxx"
)
obj = s3.get_object(Bucket=bucket, Key=arquivo)

df = pd.read_csv(obj["Body"], sep = ";", encoding= "utf-8")

print(df.columns)

Index(['CANAL', 'NR_IDENTIFICACAO', 'CLASSIFICACAO_CANAL',
       'TIPO_CONTEUDO_CANAL', 'OFERTA_CLIENTE', 'DATA_INICIO_OFERTA',
       'DENSIDADE_CANAL', 'NOME_PROGRAMADORA', 'CNPJ_PROGRAMADORA',
       'CLASSIFICACAO_PROGRAMADORA', 'PAIS_PROGRAMADORA'],
      dtype='object')


Função de String: Canal com maior quantidade de letras.

In [46]:
def letras_canais(df):
    df["TAMANHO_NOME"] = df["CANAL"].str.len()
    mais_letras = df["TAMANHO_NOME"].idxmax()
    canal_mais_letras = df.loc[mais_letras, "CANAL"]
    return canal_mais_letras
letras_canais(df)

'RDC TV - REDE DIGITAL DE COMUNICAÇÃO'

Função de data: Ano do início da oferta do canal.

In [47]:
def ano_inicio(df):   
    df["DATA_INICIO_OFERTA"] = pd.to_datetime(df["DATA_INICIO_OFERTA"], dayfirst=True, errors="coerce")
    df["ANO_OFERTA"] = df["DATA_INICIO_OFERTA"].dt.year
    mais_antigos = df.sort_values("ANO_OFERTA").head(5)
    return mais_antigos
ano_inicio(df)

,CANAL,NR_IDENTIFICACAO,CLASSIFICACAO_CANAL,TIPO_CONTEUDO_CANAL,OFERTA_CLIENTE,DATA_INICIO_OFERTA,DENSIDADE_CANAL,NOME_PROGRAMADORA,CNPJ_PROGRAMADORA,CLASSIFICACAO_PROGRAMADORA,PAIS_PROGRAMADORA,TAMANHO_NOME,ANO_OFERTA
69,CNN INTERNATIONAL,40403.30005,Canal de programação comum,Canal de conteúdo jornalístico,CANAL OFERTADO EM PACOTE,1989-01-01,PADRÃO,"TURNER INTERNATIONAL LATIN AMERICA, INC",05.583.971/0001-25,PROGRAMADORA ESTRANGEIRA,ESTADOS UNIDOS,17,1989.0
103,ESPN 2,653.30001,Canal de programação comum,Canal de conteúdo esportivo,CANAL OFERTADO EM PACOTE,1989-10-01,PADRÃO,ESPN DO BRASIL EVENTOS ESPORTIVOS LTDA.,00.637.277/0001-20,Programadora brasileira de capital estrangeiro,BRASIL,6,1989.0
56,CARTOON NETWORK,40403.30002,Canal de espaço qualificado,Canal de conteúdo infantil e adolescente,CANAL OFERTADO EM PACOTE,1993-04-30,PADRÃO,"TURNER INTERNATIONAL LATIN AMERICA, INC",05.583.971/0001-25,PROGRAMADORA ESTRANGEIRA,ESTADOS UNIDOS,15,1993.0
14,ART LATINO,14125.30002,Canal não adaptado ao mercado brasileiro,Canal de conteúdo em geral,"CANAL OFERTADO EM PACOTE, CANAL À LA CARTE",1994-07-01,PADRÃO,ALL TV COMMUNICATIONS S.A.,NaN,PROGRAMADORA ESTRANGEIRA,URUGUAI,10,1994.0
261,SONY ENTERTEINMENT TELEVISION,22335.30001,Canal de espaço qualificado,Canal de conteúdo em geral,CANAL OFERTADO EM PACOTE,1995-09-01,PADRÃO,"SET BRAZIL, LLC",NaN,PROGRAMADORA ESTRANGEIRA,ESTADOS UNIDOS,29,1995.0


Função de conversão: Canais de antes dos anos 2000

In [48]:
def canais_2000(df):
    df["ANTES_2000"] = df["ANO_OFERTA"] < 2000
    return df[["CANAL", "ANTES_2000"]].head(10)
canais_2000(df)

,CANAL,ANTES_2000
0,A&E,True
1,A&E HD,False
2,ADULT SWIM,False
3,ADULT SWIM HD,False
4,AFESP TV,False
5,AGRO CANAL,False
6,AGRO+,False
7,AGRO+ HD,False
8,AGROBRASIL TV O SEU CANAL,False
9,ALPHA CHANNEL TV,False


Função condicional: Canais de conteúdo jornalístico somente

In [49]:
def canais_jornalisticos(df):
    return df[df["TIPO_CONTEUDO_CANAL"] == "Canal de conteúdo jornalístico"][["CANAL", "TIPO_CONTEUDO_CANAL"]].head(10)
canais_jornalisticos(df)

,CANAL,TIPO_CONTEUDO_CANAL
6,AGRO+,Canal de conteúdo jornalístico
7,AGRO+ HD,Canal de conteúdo jornalístico
20,BANDNEWS,Canal de conteúdo jornalístico
21,BANDNEWS HD,Canal de conteúdo jornalístico
26,BLOOMBERG TELEVISION,Canal de conteúdo jornalístico
27,BLOOMBERG TELEVISION HD,Canal de conteúdo jornalístico
28,BM&C NEWS,Canal de conteúdo jornalístico
52,CANAL SIDYS,Canal de conteúdo jornalístico
53,CANAL TCM 10 HD,Canal de conteúdo jornalístico
66,CNN BRASIL,Canal de conteúdo jornalístico


Função de agregação: Quantidade de canais por tipo de conteúdo

In [50]:
def num_canais_conteudo(df):
    cont = df.groupby('TIPO_CONTEUDO_CANAL')['CANAL'].count()
    print("Contagem de canais por tipo de conteúdo:")
    return cont
num_canais_conteudo(df)

Contagem de canais por tipo de conteúdo:


TIPO_CONTEUDO_CANAL
Canal de conteúdo em geral                  190
Canal de conteúdo erótico                     6
Canal de conteúdo esportivo                  83
Canal de conteúdo infantil e adolescente     22
Canal de conteúdo jornalístico               25
Canal de conteúdo religioso                   2
Canal de conteúdo videomusical                8
Canal de televenda ou infomercial            11
Name: CANAL, dtype: int64

Filtrando dados usando dois operadores lógicos: Canais esportivos, avulsos e em HD

In [51]:
def filtro(df):
    filtro_canais = (df["TIPO_CONTEUDO_CANAL"] == "Canal de conteúdo esportivo") & \
                    (df["OFERTA_CLIENTE"] == "CANAL À LA CARTE") & \
                    (df["DENSIDADE_CANAL"] == "ALTA DEFINIÇÃO")
    
    return df.loc[filtro_canais, ["CANAL", "TIPO_CONTEUDO_CANAL", "OFERTA_CLIENTE", "DENSIDADE_CANAL"]]

filtro(df)

,CANAL,TIPO_CONTEUDO_CANAL,OFERTA_CLIENTE,DENSIDADE_CANAL
43,CANAL PAULISTÃO 1 - HD,Canal de conteúdo esportivo,CANAL À LA CARTE,ALTA DEFINIÇÃO
45,CANAL PAULISTÃO 2 - HD,Canal de conteúdo esportivo,CANAL À LA CARTE,ALTA DEFINIÇÃO
47,CANAL PAULISTÃO 3 - HD,Canal de conteúdo esportivo,CANAL À LA CARTE,ALTA DEFINIÇÃO
184,NORDESTE FC 1 - HD,Canal de conteúdo esportivo,CANAL À LA CARTE,ALTA DEFINIÇÃO
186,NORDESTE FC 2 - HD,Canal de conteúdo esportivo,CANAL À LA CARTE,ALTA DEFINIÇÃO
188,NORDESTE FC 3 - HD,Canal de conteúdo esportivo,CANAL À LA CARTE,ALTA DEFINIÇÃO
190,NORDESTE FC 4 - HD,Canal de conteúdo esportivo,CANAL À LA CARTE,ALTA DEFINIÇÃO
192,NORDESTE FC 5 - HD,Canal de conteúdo esportivo,CANAL À LA CARTE,ALTA DEFINIÇÃO
194,NORDESTE FC 6 - HD,Canal de conteúdo esportivo,CANAL À LA CARTE,ALTA DEFINIÇÃO
196,NORDESTE FC 7 - HD,Canal de conteúdo esportivo,CANAL À LA CARTE,ALTA DEFINIÇÃO
